In [1]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec

from nltk.tokenize import RegexpTokenizer
from nltk import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression

alpha_tokenizer = RegexpTokenizer("[A-Za-z]\\w+")
lemmatizer = WordNetLemmatizer()
stop = stopwords.words("english")




In [2]:
train = pd.read_csv("../input/train.csv")
test = pd.read_csv("../input/test.csv")




In [3]:
# Build token lists for Word2Vec training
data = [
    [
        lemmatizer.lemmatize(word.lower())
        for word in alpha_tokenizer.tokenize(sent)
        if word.lower() not in stop
    ]
    for sent in train.text.values
]

NUM_FEATURES = 200

# In gensim 4 the argument is `vector_size` instead of `size`
model = Word2Vec(
    sentences=data,
    vector_size=NUM_FEATURES,
    min_count=3,
    window=5,
    sg=1,
    alpha=1e-4,
    workers=4,
)




In [4]:
# Number of words in the vocabulary (updated API)
vocab_size = len(model.wv.key_to_index)
print("Vocabulary size:", vocab_size)




Vocabulary size: 10253


In [5]:
# Example similarity query (updated API)
try:
    similar = model.wv.most_similar("raven")
    print("Most similar to 'raven':", similar[:5])
except KeyError:
    print("'raven' not in vocabulary.")




Most similar to 'raven': [('society', 0.24269001185894012), ('scoff', 0.24252115190029144), ('calculus', 0.23626945912837982), ('chaos', 0.23506875336170197), ('prone', 0.22979415953159332)]


In [6]:
def get_feature_vec(tokens, num_features, wv_model):
    """
    Compute the average Word2Vec vector for a list of tokens.
    """
    feature_vec = np.zeros(num_features, dtype="float32")
    missed = 0
    for word in tokens:
        try:
            feature_vec += wv_model[word]
        except KeyError:
            missed += 1
            continue
    valid_cnt = len(tokens) - missed
    if valid_cnt == 0:
        return np.zeros(num_features, dtype="float32")
    return feature_vec / valid_cnt




In [7]:
vectors = []
for sentence in train.text.values:
    tokens = [
        lemmatizer.lemmatize(word.lower())
        for word in alpha_tokenizer.tokenize(sentence)
        if word.lower() not in stop
    ]
    vectors.append(get_feature_vec(tokens, NUM_FEATURES, model.wv))

vectors = np.array(vectors)  # shape (n_samples, NUM_FEATURES)




In [8]:
train["author"] = train["author"].map({"EAP": 0, "HPL": 1, "MWS": 2})




In [9]:
estimator = LogisticRegression(C=1, max_iter=1000, n_jobs=4)
estimator.fit(vectors, train.author.values)




LogisticRegression(C=1, max_iter=1000, n_jobs=4)

In [10]:
test_vectors = []
for sentence in test.text.values:
    tokens = [
        lemmatizer.lemmatize(word.lower())
        for word in alpha_tokenizer.tokenize(sentence)
        if word.lower() not in stop
    ]
    test_vectors.append(get_feature_vec(tokens, NUM_FEATURES, model.wv))

test_vectors = np.array(test_vectors)




In [11]:
probs = estimator.predict_proba(test_vectors)




In [12]:
author_df = pd.DataFrame(probs, columns=["EAP", "HPL", "MWS"])
submission = pd.DataFrame()
submission["id"] = test["id"]
submission[["EAP", "HPL", "MWS"]] = author_df[["EAP", "HPL", "MWS"]]
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")

Submission saved to submission.csv
